In [1]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

In [2]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


models to use:  ['gpt-35-turbo', 'gpt-4-vision-preview', 'dalle', 'gemini-pro', 'gemini-1.5-pro-001', 'gemini-1.5-pro', 'gemini-pro-vision']


In [3]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [4]:
config_list = [
    {
        "model": "gemini-3.8-flash",
        "api_key": "apikey",
        "api_type": "google"
    }
]

In [5]:
import logging
logging.raiseExceptions = False

import agentops
agentops.init("f9de56a1-8b24-408e-a137-7e9638f47479")  
print("AgentOps initialized successfully")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor


AgentOps initialized successfully


In [6]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)

In [7]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [8]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [ ]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
#agentops.end_session("Success")

Trying to create collection.


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-11-16 15:45:23,570 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 210 chunks.
2026-11-16 15:45:23,596 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Model gemini-3.8-flash not found. Using cl100k_base encoding.


VectorDB returns doc_ids:  [['c024e487', 'e6980caf', '5597c053', '330fa96a', 'ae7d3b76', '7b45c013', '363de674', '5385d2ca', '0acb4ca9', 'a3e8f8c3', 'ff103f25', '4a59535b', 'a1dcc20c', '8b0b85f0', '797e9c8f', 'a0f75e72', 'bfa56a62', '94c25c16', '4e6b17f7', '23d76c9a']]
Adding content of doc c024e487 to context.


Model gemini-3.8-flash not found. Using cl100k_base encoding.


Adding content of doc e6980caf to context.


Model gemini-3.8-flash not found. Using cl100k_base encoding.


... (28 outputs left out)

Adding content of doc bfa56a62 to context.


Model gemini-3.8-flash not found. Using cl100k_base encoding.


ragproxyagent (to assistant):

You're a retrieve augmented chatbot. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
You must give as short an answer as possible.

User's question is: summarize the content and use python code to any calculation given in the book

Context is: 79
THE LAW OF SACRIFICE
22 Laws of Marketing  10/31/02  12:23 PM  Page 79the company looked at the books and decided to focus
on the only product it made money on: toys. As long as
Interstate was going to focus on toys, it decided to
change its name to Toys “R” Us. Today Toys “R” Us
does 20 percent of the retail toy business in the coun-
try. Very profitably, too. In its last fiscal year, Toys “R”
Us made $326 million on sales of $5.5 billion.
Many retail chains are successfully patterning
themselves on the Toys “R” Us formula: a narrow
focus with in-depth sto

C:\Anaconda3\envs\agenticAI\Lib\site-packages\autogen\oai\gemini.py:809: UserWarning: Cost calculation is not implemented for model gemini-3.8-flash. Cost will be calculated zero.
  warnings.warn(
🖇 AgentOps: Unexpected error during span export: 'AuthenticatedOTLPExporter' object has no attribute '_headers'
